# Representação de números

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Escrever e ler números de 8 bits em complemento de dois
- Prever quando uma conta com int ou byte do Java estoura, e o que ela dá
- Explicar por que 0,1 + 0,2 não é exatamente 0,3 num double
- Escolher um jeito seguro de guardar dinheiro e de comparar decimais

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## Em resumo

Bits não têm sinal de menos, então os computadores usam **complemento de dois**: num byte, o bit mais à esquerda vale **−128** em vez de +128, e os outros mantêm seus valores. Um byte então guarda de −128 a 127, e a soma comum continua funcionando.

![O byte 11111011 em complemento de dois é −5](signed-byte.pt-br.svg)

Para inverter o sinal de um número, inverta todos os bits e some 1: 5 é `00000101`, então −5 é `11111010` + 1 = `11111011`. O `int` do Java usa 32 bits, de −2.147.483.648 a 2.147.483.647. Quando um resultado não cabe, os bits extras são descartados e ele **dá a volta**: `Integer.MAX_VALUE + 1` é `-2147483648`, e o Java não dá erro. Isso é **overflow** (estouro).

#### 🎞️ Negando em complemento de dois

![Negando 5 num byte: inverta 00000101 para 11111010, some 1 para obter 11111011, que é -128 + 123 = -5](negate-steps.pt-br.svg)

**Preveja e depois rode:** complemento de dois e o "dar a volta".

In [ ]:
IO.println(Integer.toBinaryString(-5));     // os 32 bits de -5
IO.println((byte) 200);                     // 200 não cabe num byte
IO.println(Integer.MAX_VALUE + 1);          // dá a volta
IO.println(Math.abs(Integer.MIN_VALUE));    // uma surpresa

### ✏️ Exercício: Inverta um byte

Inverta os 8 bits de um byte (um valor de 0 a 255). `~` inverte os 32 bits de um int, então fique só com os 8 de baixo usando `& 0xFF`.

In [ ]:
int invertByte(int b) {
    // TODO: inverta os bits e fique com 8 deles
    return 0;
}

void main() {
    check("5", invertByte(5), 250);
    check("0", invertByte(0), 255);
    check("255", invertByte(255), 0);
}

<details>
<summary>Ver uma resposta</summary>

```java
int invertByte(int b) {
    return ~b & 0xFF;
}

void main() {
    check("5", invertByte(5), 250);
    check("0", invertByte(0), 255);
    check("255", invertByte(255), 0);
}
```

</details>

Frações são mais difíceis. Depois da vírgula binária as posições valem ½, ¼, ⅛, … então 0,5 e 0,25 são exatos, mas **0,1 se repete para sempre** em binário, como 1/3 no decimal. Um `double` guarda o valor mais próximo que consegue, em 64 bits: um sinal, um expoente e uma fração de 52 bits, que com um 1 inicial implícito dá 53 bits de precisão, cerca de 15–16 dígitos decimais. É por isso que `0.1 + 0.2` imprime `0.30000000000000004`.

Então: compare doubles com uma tolerância, `Math.abs(a - b) < 1e-9`, e guarde dinheiro em centavos inteiros num `long`, ou em `BigDecimal`.

**0,1 não é exato em binário:**

In [ ]:
double soma = 0.1 + 0.2;
IO.println(soma);
IO.println(soma == 0.3);
IO.println(Math.abs(soma - 0.3) < 1e-9);

double total = 0;
for (int i = 0; i < 10; i++) {
    total += 0.1; // dez moedas de 10 centavos, como double
}
IO.println("dez vezes 0,1: " + total);
IO.println("em centavos:   " + (10 * 10) + " (um long não arredonda)");
IO.println("BigDecimal:    " + new BigDecimal("0.1").multiply(BigDecimal.TEN));

### ✏️ Exercício: Dinheiro em centavos

Transforme um preço em centavos inteiros sem passar por um double: `12.34` é 1234. Um preço não tem casas decimais (`12`), tem uma (`12.3`) ou duas (`12.34`). Divida no ponto.

In [ ]:
long cents(String price) {
    // TODO: divida em "\\." e combine as duas partes
    return 0;
}

void main() {
    check("12.34", cents("12.34"), 1234L);
    check("0.05", cents("0.05"), 5L);
    check("19.90", cents("19.90"), 1990L);
    check("uma casa", cents("12.3"), 1230L);
    check("sem casas", cents("12"), 1200L);
}

<details>
<summary>Ver uma resposta</summary>

```java
long cents(String price) {
    String[] parts = price.split("\\.");
    long whole = Long.parseLong(parts[0]) * 100;
    if (parts.length == 1) {
        return whole;
    }
    String twoDigits = (parts[1] + "0").substring(0, 2);   // "3" becomes "30"
    return whole + Long.parseLong(twoDigits);
}

void main() {
    check("12.34", cents("12.34"), 1234L);
    check("0.05", cents("0.05"), 5L);
    check("19.90", cents("19.90"), 1990L);
    check("uma casa", cents("12.3"), 1230L);
    check("sem casas", cents("12"), 1200L);
}
```

</details>

## Leituras

- [Twos complement: Negative numbers in binary](https://www.youtube.com/watch?v=4qH4unVtJkE) (Ben Eater, 15 min): Por que um bit de sinal e o complemento de um dão problema, e como o complemento de dois faz a soma comum funcionar também para números negativos.
- [The Floating-Point Guide: Basic answers](https://floating-point-gui.de/basic/) (floating-point-gui.de, 5 min): Por que 0,1 + 0,2 não dá 0,3, e as três saídas: arredondar ao imprimir, comparar com uma tolerância, ou usar números inteiros ou um tipo decimal. O site também tem uma página sobre Java.
- [Floating Point Numbers](https://www.youtube.com/watch?v=PZRI1IfStY0) (Computerphile, 10 min): Como o ponto flutuante funciona como a notação científica em binário, e por que algumas frações decimais nunca podem ser guardadas exatamente.

## Verifique

1. Quais são os 8 bits de −1 em complemento de dois?
2. O que `(byte) 200` dá no Java, e por quê?
3. Quais de 0,5, 0,1 e 0,75 um `double` consegue guardar exatamente?

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Complemento de dois** · Fácil
- **Lendo um byte com sinal** · Fácil
- **Detector de overflow** · Médio
- **Frações em binário** · Difícil
- Teste: **Calculadora de programador** · Difícil